# A variational autoencoder for pufferfish pictures

The same 5000 made-up pufferfish as in the autoencoder notebook, drawn by the same few lines of Python from three numbers:

* $x$ and $y$, where the fish sits in the frame, as fractions of the frame's width and height measured from its bottom left corner,
* $m$, how puffed up it is, from $0$ (a small flat fish) to $1$ (a big round ball with spikes).

The plain autoencoder squeezed every picture into three numbers of its own, and its sliders had two problems: one slider moved the fish and puffed it up at the same time, and about half of the settings decoded to a smudge instead of a fish. This notebook trains a **variational autoencoder** (VAE), the one on the last slide (Kingma and Welling, 2014), on the same fish. Its code is three numbers again, and it ends with the same sliders.

Nothing here is an exercise: run the cells from top to bottom. The cells that show only a title draw the fish and the pictures; their Show code button opens one if you are curious. Training takes a minute or two on Colab.

In [ ]:
#@title Imports { display-mode: "form" }
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import ipywidgets as widgets

## 1. Drawing a pufferfish

A picture is a grid of $32 \times 32$ numbers between $0$ (empty water) and $1$ (dark ink). `draw_fish(x, y, m)` paints a light body with a fin on each side, and on top of it a dark face: two eyes and an open mouth. Above $m = 0.5$ the body grows spikes.

In [ ]:
#@title Drawing a pufferfish { display-mode: "form" }
# Where every pixel sits: u runs from 0 at the left edge to 1 at the right edge,
# v from 0 at the bottom edge to 1 at the top edge.
u, v = np.meshgrid(np.linspace(0, 1, 32), np.linspace(1, 0, 32))


def blob(cx, cy, half_width, half_height, spikes=0):
    # An ellipse centred at (cx, cy): 1 inside and 0 outside, fading out over the
    # last quarter of the way to its rim. spikes pushes the rim further out.
    distance = np.sqrt(((u - cx) / half_width) ** 2 + ((v - cy) / half_height) ** 2)
    return np.clip(4 * (1 + spikes - distance), 0, 1)


def draw_fish(x, y, m):
    width = 0.15 + 0.07 * m       # half the width of the body
    height = 0.06 + 0.16 * m      # half its height: flat at m = 0, round at m = 1
    angle = np.arctan2(v - y, u - x)
    spikes = max(0, m - 0.5) * 0.7 * np.cos(7 * angle) ** 8   # 14 spikes, only above m = 0.5
    body = (blob(x, y, width, height, spikes)
            + blob(x - width, y, 0.06, 0.04)                        # left fin
            + blob(x + width, y, 0.06, 0.04))                       # right fin
    face = (blob(x - 0.4 * width, y + 0.3 * height, 0.03, 0.04)     # left eye
            + blob(x + 0.4 * width, y + 0.3 * height, 0.03, 0.04)   # right eye
            + blob(x, y - 0.4 * height, 0.04, 0.03))                # open mouth
    return 0.4 * np.clip(body, 0, 1) + 0.6 * np.clip(face, 0, 1)

## 2. Five thousand fish

We draw $x$, $y$ and $m$ at random and paint one fish for each. The autoencoder will only ever see the pictures; we keep the three numbers behind each one, `truth`, to label the pictures for us.

In [ ]:
#@title Five thousand fish { display-mode: "form" }
rng = np.random.default_rng(seed=0)   # seeded: the same 5000 fish on every computer

# One row (x, y, m) per fish: x and y between 0.25 and 0.75, m between 0 and 1.
truth = rng.uniform(low=[0.25, 0.25, 0], high=[0.75, 0.75, 1], size=(5000, 3))

pictures = np.array([draw_fish(x, y, m) for x, y, m in truth])
pictures = torch.tensor(pictures, dtype=torch.float32)
print(pictures.shape)

In [ ]:
#@title Twelve of the fish { display-mode: "form" }
def show(ax, picture, title):
    ax.imshow(picture, cmap="Oranges", vmin=0, vmax=1)
    ax.set(xticks=[], yticks=[], title=title)


fig, axes = plt.subplots(2, 6, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    x, y, m = truth[i]
    show(ax, pictures[i], f"fish {i}\nx={x:.2f}, y={y:.2f}\nm={m:.2f}")
plt.show()

## 3. The variational autoencoder

The decoder is the plain autoencoder's, line for line: from $3$ numbers up to $1024$, one per pixel. So is the encoder, except that its last layer gives $6$ numbers instead of $3$, two for each number of the code:

* the **centre** $z$, the code of the picture,
* the **radius** $\sigma$, how far from the centre the decoder's point may stray in training.

The encoder gives each radius as $\log \sigma^2$, a number that may come out positive or negative; $\sigma = e^{\frac{1}{2} \log \sigma^2}$ is then always positive. On the slides the encoder is $\varphi$ and the decoder is $\varphi^{-1}$.

In [ ]:
torch.manual_seed(0)   # seeded: the same starting weights on every computer

encoder = nn.Sequential(
    nn.Flatten(),                           # 32 x 32 pixels into one list of 1024 numbers
    nn.Linear(1024, 512), nn.ReLU(),
    nn.Linear(512, 128), nn.ReLU(),
    nn.Linear(128, 6),                      # 6 numbers: the centre z (3) and log sigma squared (3)
)

decoder = nn.Sequential(
    nn.Linear(3, 128), nn.ReLU(),
    nn.Linear(128, 512), nn.ReLU(),
    nn.Linear(512, 1024),                   # 1024 numbers, one per pixel
    nn.Unflatten(1, (32, 32)),              # folded back into 32 x 32 pixels
)

## 4. Training

One training step works on $64$ fish picked at random, as on the slide:

1. The encoder gives every fish a centre $z$ and a radius $\sigma$ for each of its three numbers.
2. The decoder gets a random point $\tilde z = z + \sigma \cdot \varepsilon$ near the centre, where $\varepsilon$ is three random numbers from a bell curve (a standard normal distribution), and must still rebuild the fish. Drawn this way, the random point still lets the gradient reach $z$ and $\sigma$; this is called the reparameterization trick.
3. The loss has two terms:
    * the **miss**: the squared differences between the rebuilt pixels and the original ones, **summed** over the $1024$ pixels of a fish (the plain autoencoder averaged them),
    * the **second term**, the KL term: $\frac{1}{2}\left(z^2 + \sigma^2 - 1 - \log \sigma^2\right)$, added up over the three numbers. It is $0$ when a centre sits at $0$ with a radius of $1$, and grows as a centre moves away from $0$ or a radius away from $1$. It pulls the codes of all pictures into one round cloud.

The loss is the miss plus a **weight** times the second term. Kingma and Welling's loss has weight $1$ (for squared misses, it is the loss of a decoder that expects noise with a variance of $\frac{1}{2}$ in every pixel). Training runs in two stages:

* **Stage 1.** The decoder gets numbers 1 and 2 only, number 3 is held at $0$, and the weight is 1.25, a little more than $1$, to press the two numbers harder to take one job each. With two numbers for a fish, the VAE spends them on what changes a picture most: where the fish is, sideways and up and down.
* **Stage 2.** The decoder gets all three numbers and the weight drops to 0.5. The one thing left to describe is how puffed up the fish is, and number 3 takes it. The lower weight lets every centre be more precise, so the fish come out sharper.

Why not train all three numbers at once? We tried, from many random starts. With weight $1$ from the first step, 4 of 9 runs did not split into $x$, $y$ and $m$ one each. Mostly the VAE had stored the puff as a distance from the centre of a plane of two numbers, the way a clock hand needs a direction and a length, and no single slider puffed the fish. Holding number 3 back is a nudge in the training. Nothing in this notebook ever shows the VAE an $x$, a $y$ or an $m$.

Both loops are the one of weekend 1: while the loss is too big, take a step. One batch's loss jumps up and down from step to step, so the loops follow its **running average**: after every step, the average moves $1\%$ of the way towards the newest batch's loss. Stage 1 ends when it drops below $10.4$, stage 2 when it drops below $6.9$, after about 2000 steps in all for most random starts, which takes a minute or two on Colab.

In [ ]:
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)


def training_step(weight, third):
    # One step on 64 fish picked at random; third = 0 holds number 3 at 0.
    batch = pictures[torch.randint(0, len(pictures), (64,))]
    z, log_var = encoder(batch).chunk(2, dim=1)      # per fish: 3 centres and 3 values of log sigma squared
    sigma = torch.exp(0.5 * log_var)                 # the 3 radii
    z_tilde = z + sigma * torch.randn_like(z)        # a random point near the centre
    z_tilde = z_tilde * torch.tensor([1.0, 1.0, third])
    miss = ((decoder(z_tilde) - batch) ** 2).sum(dim=(1, 2)).mean()       # summed over the 1024 pixels of a fish
    kl = (0.5 * (z ** 2 + sigma ** 2 - 1 - log_var)).sum(dim=1).mean()    # the second term
    batch_loss = miss + weight * kl
    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()
    return batch_loss.item()


loss = 20.0                                          # the running average of the loss, any start above the thresholds

# Stage 1: numbers 1 and 2 only
while loss > 10.4:
    loss = 0.99 * loss + 0.01 * training_step(weight=1.25, third=0.0)
    print(f"\rstage 1, average loss {loss:.3f}", end="")
print()

# Stage 2: all three numbers
while loss > 6.9:
    loss = 0.99 * loss + 0.01 * training_step(weight=0.5, third=1.0)
    print(f"\rstage 2, average loss {loss:.3f}", end="")

## 5. What does each number do?

Each row of pictures moves one of the three numbers alone, from $-2.5$ to $2.5$, with the other two held at $0$, the centre of the cloud. If the VAE has given each number one job, a row only moves the fish sideways, only moves it up and down, or only puffs it up.

The table under the pictures measures the same thing on all 5000 fish: how closely each number of the code follows $x$, $y$ and $m$. It is the Spearman correlation: $1$ or $-1$ means the number rises (or falls) every time the factor rises, $0$ means it ignores it. Nobody told the VAE about $x$, $y$ and $m$, so which number takes which job, if any, is up to the training: the table is measured on this very run.

In [ ]:
#@title What each number does { display-mode: "form" }
def follows_what(codes):
    # The Spearman correlation of every number of the code with x, y and m: the
    # ordinary correlation of their ranks. A number follows a factor when that
    # correlation is at least 0.8 (or at most -0.8) and the other two stay
    # within 0.3 of 0; the answer counts only if the three numbers follow
    # three different factors.
    code_ranks = codes.numpy().argsort(axis=0).argsort(axis=0)
    truth_ranks = truth.argsort(axis=0).argsort(axis=0)
    table = np.array([[np.corrcoef(code_ranks[:, j], truth_ranks[:, k])[0, 1] for k in (0, 1, 2)]
                      for j in (0, 1, 2)])
    follows = {}
    for j in (0, 1, 2):
        strong = [k for k in (0, 1, 2) if abs(table[j, k]) >= 0.8]
        if len(strong) == 1 and all(abs(table[j, k]) < 0.3 for k in (0, 1, 2) if k != strong[0]):
            follows[j] = "xym"[strong[0]]
    if sorted(follows.values()) != ["m", "x", "y"]:
        follows = {}
    return table, follows


codes = encoder(pictures).detach()[:, :3]   # the centre z of every fish: its three numbers

fig, axes = plt.subplots(3, 9, figsize=(16, 6.2))
for row in (0, 1, 2):
    for col, value in enumerate(np.linspace(-2.5, 2.5, 9)):
        numbers = torch.zeros(1, 3)
        numbers[0, row] = value
        show(axes[row, col], decoder(numbers).detach()[0], f"number {row + 1} = {value:.1f}")
plt.show()

table, follows = follows_what(codes)
print("Spearman correlation over all 5000 fish")
print("              x      y      m")
for j in (0, 1, 2):
    print(f"number {j + 1}  " + "  ".join(f"{c:5.2f}" for c in table[j]))
if follows:
    print("\nEach number follows one factor: " + ", ".join(f"number {j + 1} follows {follows[j]}" for j in (0, 1, 2)) + ".")
else:
    print("\nIn this run the three numbers do not split into x, y and m one each: at least one of them mixes two factors or follows none.")

## 6. Play with the three numbers

Pick a fish with the first slider. On the left is its picture, with the $x$, $y$ and $m$ it was drawn from and its three numbers, the centre $z$ the encoder gives it. The three sliders below start at those numbers, and on the right is what the decoder draws from them. Move the sliders and a new fish appears. As in the autoencoder notebook, every slider runs from the smallest to the largest value its number takes over the 5000 fish. When part 5 found one factor per number, its name is next to the slider. After training again, run this cell again too: the sliders keep the numbers of the model they were built with.

Two things to look for:

* Does each slider do one thing, move the fish sideways, move it up and down, or puff it up? Watch the ends of the sideways and the up and down sliders: there the puff changes too, because the outer edges of the cloud hold mostly puffy fish or mostly flat ones, and the decoder draws what it saw there.
* Every setting of the sliders should give a fish, also settings that no picture was encoded to: in training the decoder saw random points around every centre, and the second term packed the centres into one cloud, so it has learned to draw a fish everywhere in between. The fish are blurrier than the plain autoencoder's, though: the decoder learns to draw what fits all the points round a centre, and the faces and the spikes fade.

In [ ]:
#@title The sliders { display-mode: "form" }
codes = encoder(pictures).detach()[:, :3]   # the centre z of every fish: its three numbers
table, follows = follows_what(codes)        # which factor each number follows, if any (part 5)

low = codes.min(dim=0).values.tolist()      # the smallest value of each number
high = codes.max(dim=0).values.tolist()     # the largest value of each number

fish = widgets.IntSlider(min=0, max=len(pictures) - 1, description="fish", continuous_update=False)
names = [f"number {j + 1}" + (f" ({follows[j]})" if j in follows else "") for j in (0, 1, 2)]
wide = {"description_width": "initial"}
number1 = widgets.FloatSlider(min=low[0], max=high[0], step=0.01, description=names[0], style=wide)
number2 = widgets.FloatSlider(min=low[1], max=high[1], step=0.01, description=names[1], style=wide)
number3 = widgets.FloatSlider(min=low[2], max=high[2], step=0.01, description=names[2], style=wide)


def pick_fish(change):
    # A new fish was picked: move the three sliders to its three numbers. Holding
    # their notifications sets all three before anything is redrawn, so that no
    # picture mixes the new fish's first number with the old fish's other two.
    with (number1.hold_trait_notifications(),
          number2.hold_trait_notifications(),
          number3.hold_trait_notifications()):
        number1.value, number2.value, number3.value = codes[fish.value].tolist()


def draw(fish, number1, number2, number3):
    x, y, m = truth[fish]
    a, b, c = codes[fish].tolist()
    new_picture = decoder(torch.tensor([[number1, number2, number3]])).detach()[0]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4.8))
    show(left, pictures[fish],
         f"fish {fish}: x={x:.2f} y={y:.2f} m={m:.2f}\nencoded as {a:.2f}, {b:.2f}, {c:.2f}")
    show(right, new_picture,
         f"decoded from\n{number1:.2f}, {number2:.2f}, {number3:.2f}")
    plt.show()


fish.observe(pick_fish, names="value")
pick_fish(None)   # start on fish 0
widgets.interact(draw, fish=fish, number1=number1, number2=number2, number3=number3);